# 1. 패키지 설치

In [1]:
%pip install python-dotenv langchain langchain-upstage langchain-community langchain-text-splitters docx2txt langchain-chroma


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# 2. Knowledge Base 구성을 위한 데이터 생성

- [RecursiveCharacterTextSplitter](https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/)를 활용한 데이터 chunking
    - split 된 데이터 chunk를 Large Language Model(LLM)에게 전달하면 토큰 절약 가능
    - 비용 감소와 답변 생성시간 감소의 효과
    - LangChain에서 다양한 [TextSplitter](https://python.langchain.com/v0.2/docs/how_to/#text-splitters)들을 제공
- `chunk_size` 는 split 된 chunk의 최대 크기
- `chunk_overlap`은 앞 뒤로 나뉘어진 chunk들이 얼마나 겹쳐도 되는지 지정

In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

/var/folders/jw/qym3n0892n51wr3xcmd925d00000gn/T/ipykernel_33565/437428471.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [3]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

# 환경변수를 불러옴
load_dotenv()

# OpenAI에서 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

In [4]:
from langchain_chroma import Chroma

# 데이터를 처음 저장할 때 
database = Chroma.from_documents(documents=document_list, embedding=embedding, collection_name='chroma-tax', persist_directory="./chroma")

# 이미 저장된 데이터를 사용할 때 
# database = Chroma(collection_name='chroma-tax', persist_directory="./chroma", embedding_function=embedding)

# 3. 답변 생성을 위한 Retrieval

- `Chroma`에 저장한 데이터를 유사도 검색(`similarity_search()`)를 활용해서 가져옴

In [27]:
query = '부양가족에 대한 정의는 무엇인가?'

# `k` 값을 조절해서 얼마나 많은 데이터를 불러올지 결정
retrieved_docs = database.similarity_search(query, k=10)

In [28]:
retrieved_docs

[Document(id='4d8392e4-6670-4b91-a38e-128f32d501f8', metadata={'source': './tax.docx'}, page_content='5. 주택에 대한 「부동산 가격공시에 관한 법률」에 따른 개별주택가격 및 공동주택가격이 공시되기 전에 차입한 경우에는 차입일 이후 같은 법에 따라 최초로 공시된 가격을 해당 주택의 기준시가로 본다.\n\n⑥ 제5항 단서에도 불구하고 장기주택저당차입금이 다음 각 호의 어느 하나에 해당하는 경우에는 연 800만원 대신 그 해당 각 호의 금액을 공제한도로 하여 제5항 본문을 적용한다.<신설 2014. 12. 23., 2023. 12. 31.>\n\n1. 차입금의 상환기간이 15년 이상인 장기주택저당차입금의 이자를 대통령령으로 정하는 고정금리 방식(이하 이 항에서 “고정금리”라 한다)으로 지급하고, 그 차입금을 대통령령으로 정하는 비거치식 분할상환 방식(이하 이 항에서 “비거치식 분할상환”이라 한다)으로 상환하는 경우: 2천만원\n\n2. 차입금의 상환기간이 15년 이상인 장기주택저당차입금의 이자를 고정금리로 지급하거나 그 차입금을 비거치식 분할상환으로 상환하는 경우: 1천800만원\n\n3. 차입금의 상환기간이 10년 이상인 장기주택저당차입금의 이자를 고정금리로 지급하거나 그 차입금을 비거치식 분할상환으로 상환하는 경우: 600만원\n\n⑦ 삭제<2014. 1. 1.>\n\n⑧ 제1항ㆍ제4항 및 제5항에 따른 공제는 해당 거주자가 대통령령으로 정하는 바에 따라 신청한 경우에 적용하며, 공제액이 그 거주자의 해당 과세기간의 합산과세되는 종합소득금액을 초과하는 경우 그 초과하는 금액은 없는 것으로 한다.<개정 2012. 1. 1., 2014. 1. 1.>\n\n1. 삭제<2014. 1. 1.>\n\n2. 삭제<2014. 1. 1.>\n\n⑨ 삭제<2014. 1. 1.>\n\n⑩ 제1항ㆍ제4항ㆍ제5항 및 제8항에 따른 공제를 “특별소득공제”라 한다.<개정 2014. 1. 1.>\n\n⑪ 특별

# 4. Augmentation을 위한 Prompt 활용

- Retrieval된 데이터는 LangChain에서 제공하는 프롬프트(`"rlm/rag-prompt"`) 사용

In [7]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage()

In [17]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("human",
     "You are an assistant for question-answering tasks. "
     "Use the following retrieved context to answer the question. "
     "If you don't know the answer, just say that you don't know. "
     "Use three sentences maximum and keep the answer concise.\n\n"
     "Question: {question}\nContext: {context}\nAnswer:")
])

# 5. 답변 생성

- [RetrievalQA](https://docs.smith.langchain.com/old/cookbook/hub-examples/retrieval-qa-chain)를 통해 LLM에 전달
    - `RetrievalQA`는 [create_retrieval_chain](https://python.langchain.com/v0.2/docs/how_to/qa_sources/#using-create_retrieval_chain)으로 대체됨
    - 실제 ChatBot 구현 시 `create_retrieval_chain`으로 변경하는 과정을 볼 수 있음

In [19]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

In [20]:
ai_message = qa_chain({"query": query})

/var/folders/jw/qym3n0892n51wr3xcmd925d00000gn/T/ipykernel_33565/3455095564.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  ai_message = qa_chain({"query": query})


In [21]:
# 강의에서는 위처럼 진행하지만 업데이트된 LangChain 문법은 `.invoke()` 활용을 권장
ai_message = qa_chain.invoke({"query": query})

In [22]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '제공된 문맥에는 연봉 5천만원인 직장인의 소득세를 직접 계산하는 정보는 없습니다. 그러나 일반적으로 소득세는 과세표준에 세율을 적용하여 계산하며, 과세표준은 각종 공제(근로소득공제, 연금소득공제, 퇴직소득공제 등)를 적용한 후에 산출됩니다. 따라서 연봉 5천만원인 직장인의 소득세를 정확히 계산하려면 해당 개인의 공제내역과 세율을 알아야 합니다. 정확한 답변을 위해서는 국세청 홈택스 등 공인된 소득세 계산 서비스를 이용하시길 권장합니다.'}